# Assignment 5 - Pandas Data Analysis
**Student:** Kanshar Maksotov  
**Group:** Po 3-23

---
## Section 1 — Load & profile

Loaded both files and checked shape, types, missing values, duplicates, and country values.

In [20]:
import pandas as pd

sales = pd.read_csv('sales_messy.csv')
customers = pd.read_csv('customers.csv')

print("=== sales_messy.csv ===")
print("Shape:", sales.shape)


=== sales_messy.csv ===
Shape: (208, 9)


In [21]:
sales.info()


<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     208 non-null    int64  
 1   order_date   208 non-null    str    
 2   customer_id  200 non-null    float64
 3   country      208 non-null    str    
 4   category     208 non-null    str    
 5   product      208 non-null    str    
 6   quantity     208 non-null    int64  
 7   unit_price   195 non-null    float64
 8   discount     188 non-null    float64
dtypes: float64(3), int64(2), str(4)
memory usage: 14.8 KB


In [22]:
print("Missing values per column:")
print(sales.isnull().sum())


Missing values per column:
order_id        0
order_date      0
customer_id     8
country         0
category        0
product         0
quantity        0
unit_price     13
discount       20
dtype: int64


In [23]:
print("Duplicate rows:", sales.duplicated().sum())


Duplicate rows: 8


In [24]:
print("country.unique():")
print(sales['country'].unique())


country.unique():
<StringArray>
[     'GERMANY',      'Germany',       'France',      ' France',
   'Kazakhstan',           'UK', ' kazakhstan ',          'uk ',
       'Poland',          'usa',          'USA',       'Russia']
Length: 12, dtype: str


**Found Problems:**

1. **Missing values:** `customer_id` — 8 rows, `unit_price` — 13 rows, `discount` — 20 rows.
2. **Duplicate rows:** 8 exact duplicates that inflate counts.
3. **Inconsistent country values:** `'GERMANY'` vs `'Germany'`, `' France'` (leading space), `' kazakhstan '` (both spaces and wrong case), `'uk '` (trailing space) — same country stored multiple ways.
4. **`order_date`** is dtype `str` (object), not `datetime` — can't use for time analysis yet.
5. **`customer_id`** is `float64` due to NaN rows — should be integer after we drop nulls.


---
## Section 2 — Clean

Cleaned the data step by step — removed duplicates, fixed country values, filled missing fields, and parsed dates.
After cleaning, confirmed zero missing values in the key columns.


In [25]:
# remove duplicates
df = sales.drop_duplicates()
print("After drop_duplicates:", df.shape)


After drop_duplicates: (200, 9)


In [26]:
# standardise country
df['country'] = df['country'].str.strip().str.title()
print("Unique countries after standardisation:", df['country'].unique())


Unique countries after standardisation: <StringArray>
['Germany', 'France', 'Kazakhstan', 'Uk', 'Poland', 'Usa', 'Russia']
Length: 7, dtype: str


In [27]:
# fill missing discount (with 0)
df['discount'] = df['discount'].fillna(0)

# fill unit_price with median
price_median = df['unit_price'].median()
print(f"Median unit_price used for fill: {price_median}")
df['unit_price'] = df['unit_price'].fillna(price_median)


Median unit_price used for fill: 329.0


In [28]:
# drop rows with no customer_id
df = df.dropna(subset=['customer_id'])
df['customer_id'] = df['customer_id'].astype(int)
print("Shape after dropping null customer_id rows:", df.shape)


Shape after dropping null customer_id rows: (193, 9)


In [29]:
# parse order_date to datetime
df['order_date'] = pd.to_datetime(df['order_date'])
print("order_date dtype:", df['order_date'].dtype)


order_date dtype: datetime64[us]


In [30]:
# zero missing values in cleaned columns
target_cols = ['customer_id', 'order_date', 'unit_price', 'discount']
missing_after = df[target_cols].isnull().sum()
print("Missing values after cleaning:")
print(missing_after)
assert missing_after.sum() == 0, "Still have missing values — check cleaning steps!"
print("✓ All zeros — cleaning complete.")


Missing values after cleaning:
customer_id    0
order_date     0
unit_price     0
discount       0
dtype: int64
✓ All zeros — cleaning complete.


---
## Section 3 — Enrich

Added a revenue column and a month column for further analysis.


In [31]:
df['revenue'] = df['quantity'] * df['unit_price'] * (1 - df['discount'])
df['month']   = df['order_date'].dt.to_period('M')

print("New columns added:")
print(df[['order_date', 'month', 'quantity', 'unit_price', 'discount', 'revenue']].head(6))


New columns added:
  order_date    month  quantity  unit_price  discount   revenue
1 2025-07-24  2025-07         3       59.99      0.10   161.973
2 2025-02-16  2025-02         1      799.00      0.05   759.050
3 2025-12-15  2025-12         4      899.00      0.20  2876.800
4 2025-08-28  2025-08         5      549.00      0.10  2470.500
5 2025-02-28  2025-02         6       59.99      0.15   305.949
6 2025-08-11  2025-08         2        9.99      0.00    19.980


---
## Section 4 — Merge

Joined customer data to get segment and name. Checked that row count stayed the same after the merge.


In [32]:
rows_before = len(df)

merged = df.merge(
    customers[['customer_id', 'customer_name', 'segment']],
    on='customer_id',
    how='left'
)

rows_after = len(merged)
print(f"Rows before merge: {rows_before}")
print(f"Rows after  merge: {rows_after}")
assert rows_before == rows_after, "Row count changed — merge created duplicates or dropped rows!"
print("✓ Row count unchanged.")


Rows before merge: 193
Rows after  merge: 193
✓ Row count unchanged.


In [33]:
print(merged[['order_id', 'customer_id', 'customer_name', 'segment', 'revenue']].head(5))


   order_id  customer_id customer_name   segment   revenue
0      1002           28   Customer 28  Consumer   161.973
1      1003           33   Customer 33  Business   759.050
2      1004           11   Customer 11  Business  2876.800
3      1005           25   Customer 25  Consumer  2470.500
4      1006            1   Customer 01  Consumer   305.949


---
## Section 5 — Aggregate

Answered three business questions — revenue by category, by month, and by customer segment.


### 1 — Revenue per Category


In [34]:
total_revenue = merged['revenue'].sum()

rev_cat = (
    merged.groupby('category')['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'revenue': 'total_revenue'})
)
rev_cat['share_%'] = (rev_cat['total_revenue'] / total_revenue * 100).round(2)
rev_cat['total_revenue'] = rev_cat['total_revenue'].round(2)
print(rev_cat.to_string(index=False))


   category  total_revenue  share_%
    Laptops      161187.40    54.97
     Phones       63403.40    21.62
   Monitors       58295.55    19.88
Accessories       10323.55     3.52


### 2 — Revenue per Month


In [35]:
rev_month = (
    merged.groupby('month')['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'revenue': 'total_revenue'})
)
rev_month['total_revenue'] = rev_month['total_revenue'].round(2)
print(rev_month.to_string(index=False))


  month  total_revenue
2025-07       42529.43
2025-10       33697.75
2025-08       30827.43
2025-04       26456.24
2025-06       24754.84
2025-05       23633.51
2025-12       22739.75
2025-03       19836.59
2025-02       19631.08
2025-11       19117.39
2025-01       15348.40
2025-09       14637.50


### 3 — Revenue per Customer Segment


In [36]:
rev_seg = (
    merged.groupby('segment')['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'revenue': 'total_revenue'})
)
rev_seg['share_%'] = (rev_seg['total_revenue'] / total_revenue * 100).round(2)
rev_seg['total_revenue'] = rev_seg['total_revenue'].round(2)
print(rev_seg.to_string(index=False))


  segment  total_revenue  share_%
 Consumer      174850.84    59.63
Education       77782.03    26.53
 Business       40577.03    13.84


---
## Section 6 — Conclusions

Key findings from the analysis, each backed by a number from the tables above.

---

**Finding 1 — Laptops dominate revenue.**  
The Laptops category generated **161 187.40** in revenue, accounting for **54.97 %** of total revenue (293 209.90). The next category, Phones, earned 63 403.40 — less than half of Laptops. This concentration means the business is heavily dependent on one product line.

**Finding 2 — July 2025 was the strongest month.**  
Monthly revenue peaked in July 2025 at **42 529.43**, roughly 1.4× the second-best month (October: 33 697.75). A seasonal or campaign-driven spike in mid-year is worth investigating.

**Finding 3 — Consumer segment drives most revenue.**  
Consumer customers produced **174 850.84** (59.6 % of total), compared with Education (77 782.03, 26.5 %) and Business (40 577.03, 13.8 %). Despite typically having smaller budgets, individual consumers collectively outspend business accounts.

**Finding 4 — Accessories are a very minor revenue source.**  
Accessories contributed only **10 323.55** — just 3.52 % of total revenue — despite likely having many transactions. This suggests either very low unit prices, small order sizes, or both. Accessories may serve mainly as add-ons rather than a primary revenue driver.

**Finding 5 — 8 rows (3.9 % of raw data) were lost to data quality issues.**  
We removed 8 duplicates and 8 rows with no `customer_id` (16 rows, ~7.7 %), leaving 193 clean rows from the original 208. The impact on aggregate revenue is hard to quantify because missing `customer_id` rows cannot be attributed to a segment — a reminder that upstream data quality directly affects analysis completeness.
